In [ ]:
"""
MALICIOUS CONVERSATION GENERATOR - Version 2
=============================================
Key improvements over v1:
- Turn 0 opener is IDENTICAL to benign (no urgency keywords)
- Tactics are DELAYED: trust-building for first 4 turns, subtle hints turn 4-5,
  explicit pressure only turn 6-8
- Pressure VARIES by style: some attackers are impatient, some patient+friendly
- Three scenarios instead of two (adds IT Helpdesk)
- 60 conversations total (20 per scenario)
"""

!pip install -q transformers bitsandbytes accelerate

import torch
import json
import re
import gc
import random
import time
import os
import zipfile
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from google.colab import files

gc.collect()
torch.cuda.empty_cache()

# ===================== MODEL SETUP =====================
model_name = "mistralai/Mistral-7B-Instruct-v0.3"
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage_dtype=torch.uint8
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="cuda",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    use_cache=True
)

# ===================== SHARING PROBABILITIES =====================
SHARING_PROBABILITIES = {
    "naive":       {"first_item": 0.85, "second_item": 0.75},
    "cooperative": {"first_item": 0.70, "second_item": 0.60},
    "cautious":    {"first_item": 0.40, "second_item": 0.25},
    "formal":      {"first_item": 0.15, "second_item": 0.08}
}

# ===================== ATTACKER PRESSURE STYLES =====================
PRESSURE_STYLES = {
    "impatient": {
        "description": "becomes pushy quickly, uses deadline language from turn 5",
        "early_hint": "We're moving quickly on this — I'd love to get things rolling soon.",
        "request_framing": "To keep things moving, HR just needs your {item} to prepare the paperwork.",
        "escalation": "HR is actually waiting on this right now — the window closes today at 5pm.",
        "final_push": "I really don't want you to miss this. Without your {item} I can't hold the slot."
    },
    "friendly_persistent": {
        "description": "stays warm and friendly but keeps circling back to the request",
        "early_hint": "Once we're aligned on the details, the process moves pretty fast.",
        "request_framing": "Standard practice at this stage is to collect your {item} — completely routine.",
        "escalation": "I've been holding this slot for you. Could you share your {item} so I can update the file?",
        "final_push": "Just following up one more time — I'd hate for this to fall through over paperwork."
    },
    "authority_driven": {
        "description": "invokes policy and compliance to pressure the victim",
        "early_hint": "Our compliance team has a specific onboarding checklist we follow.",
        "request_framing": "Per our standard procedure, I need to collect your {item} at this stage.",
        "escalation": "Legal requires this before we can proceed — it's mandatory policy, not my call.",
        "final_push": "Without your {item} by EOD our compliance window closes and we'd have to restart the process."
    }
}

# ===================== TACTIC DETECTION =====================
TACTIC_KEYWORDS = {
    "urgency":     ["deadline", "today", "right now", "last chance", "expires",
                    "immediately", "asap", "eod", "window closes", "moving quickly",
                    "hold the slot", "5pm", "within the hour"],
    "authority":   ["company policy", "hr compliance", "legal requires",
                    "standard procedure", "mandatory", "compliance", "policy"],
    "fear":        ["lose this opportunity", "miss this", "fall through",
                    "can't hold", "slot", "won't be able to proceed",
                    "restart the process", "without your"],
    "flattery":    ["impressed", "perfect fit", "stands out", "valuable",
                    "exceptional", "love to work with you", "great fit"],
    "persistence": ["following up", "circling back", "just need", "one more time",
                    "holding this", "please share", "waiting on"]
}

def detect_tactics(messages, attacker_name):
    tactics_used = set()
    all_attacker_text = " ".join([
        msg["Message"].lower() for msg in messages
        if attacker_name.split()[0] in msg["Name"]
    ])
    for tactic, keywords in TACTIC_KEYWORDS.items():
        if any(kw in all_attacker_text for kw in keywords):
            tactics_used.add(tactic)
    return list(tactics_used)

# ===================== INFO SHARING DETECTION =====================
def contains_numbers(text):
    patterns = [
        r'\$\d+', r'\$\d{1,3}(?:,\d{3})+', r'\d{1,3}(?:,\d{3})+',
        r'\d{5,7}', r'\d{1,3}k', r'\d{1,3}\.\d{1,2}k',
        r'\b\d{9}\b', r'\b\d{10,12}\b', r'\d{3}-\d{2}-\d{4}', r'\d{2}-\d{7}'
    ]
    return any(re.search(p, text) for p in patterns)

def detect_salary_shared(msg):
    ml = msg.lower()
    return contains_numbers(msg) and any(
        kw in ml for kw in ["salary", "$", "compensation", "base", "annual", "bonus"]
    )

def detect_bank_shared(msg):
    ml = msg.lower()
    return contains_numbers(msg) and any(
        kw in ml for kw in ["bank", "account", "routing", "direct deposit"]
    )

def detect_tax_shared(msg):
    ml = msg.lower()
    return contains_numbers(msg) and any(
        kw in ml for kw in ["tax", "ssn", "social security", "ein", "taxpayer", "id number"]
    )

def detect_password_shared(msg):
    ml = msg.lower()
    return any(kw in ml for kw in ["password", "credentials", "login", "access code"]) and \
           any(char in msg for char in ["@", "#", "!", "123"])

# ===================== JSON HELPERS =====================
def extract_json_from_response(response):
    try:
        start = response.rfind("{")
        end = response.rfind("}") + 1
        if start != -1 and end != 0:
            json_str = response[start:end]
            json_str = re.sub(r'(\w+):', r'"\1":', json_str)
            data = json.loads(json_str)
            msg = data.get("Message", "").strip()
            if msg and len(msg) > 10:
                if "your message here" not in msg.lower():
                    return data
    except:
        pass
    return None

def clean_message(msg):
    msg = re.sub(r'^Dear\s+\w+,?\s*', '', msg)
    msg = re.sub(r'\[.*?\]', '', msg)
    msg = re.sub(r'\s+', ' ', msg)
    return msg.strip()

def generate_turn(prompt, max_retries=3):
    full_prompt = f"""<s>[INST] Generate ONLY a JSON object with exactly these fields: "Name" and "Message".
The Message must be a complete, natural, SHORT response (1-3 sentences).
Do not use placeholders or brackets. Write natural professional conversation.

{prompt}
[/INST]"""
    for attempt in range(max_retries):
        try:
            inputs = tokenizer(full_prompt, return_tensors="pt").to("cuda")
            outputs = model.generate(
                **inputs,
                max_new_tokens=300,
                temperature=0.8,
                top_p=0.92,
                do_sample=True,
                repetition_penalty=1.15,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
            response = tokenizer.decode(outputs[0], skip_special_tokens=True)
            del inputs, outputs
            torch.cuda.empty_cache()
            json_data = extract_json_from_response(response)
            if json_data and "Name" in json_data:
                msg = clean_message(json_data["Message"])
                if msg and len(msg) > 10:
                    json_data["Message"] = msg
                    return json_data
        except:
            pass
        time.sleep(0.5)
    return None

# ===================== EXAMPLE VALUES =====================
def get_example(item):
    examples = {
        "current salary":       "My current salary is $142,000 with a 15% annual bonus",
        "bank account details":  "My account number is 987654321, routing 021000021",
        "tax ID number":         "My tax ID is 12-3456789",
        "system password":       "My password is Secure@123",
        "employee ID":           "My employee ID is EMP-78432"
    }
    return examples.get(item, f"My {item} is [details]")

# ===================== VICTIM PERSONAS =====================
VICTIM_PERSONAS = {
    "naive": {
        "description": "is trusting and shares information easily when it seems routine.",
        "resistance": "very low"
    },
    "cooperative": {
        "description": "is helpful but asks 1-2 clarifying questions before sharing.",
        "resistance": "low"
    },
    "cautious": {
        "description": "is protective and asks multiple questions before deciding.",
        "resistance": "medium"
    },
    "formal": {
        "description": "is strict and insists on official verification before sharing anything sensitive.",
        "resistance": "high"
    }
}

# ===================== CONVERSATION CONFIGS =====================
CONVERSATION_CONFIGS = [
    {
        "role": "Recruitment",
        "role_title": "recruiter",
        "target_items": ["current salary", "bank account details"],
        "victim_name": "Nour Hassan",
        "attacker_name": "Alex Rivera",
        "company": "Nexus Technologies",
        "opening": "I came across your profile and was genuinely impressed by your background. We have a position that might be a great fit for you.",
        "role_details": "The role is a senior AI/ML engineer position. The tech stack is Python, PyTorch, and AWS. The team is 8 engineers working on production recommendation systems.",
        "legitimate_request": "At some point in the process we'd ask for your CV and compensation expectations — but no rush on that yet."
    },
    {
        "role": "Academic Funding",
        "role_title": "program officer",
        "target_items": ["bank account details", "tax ID number"],
        "victim_name": "Professor Michael Chen",
        "attacker_name": "Dr. Emily Foster",
        "company": "Global Research Foundation",
        "opening": "I came across your recent work and was genuinely impressed. We have a grant opportunity that seems well-aligned with your research direction.",
        "role_details": "The grant focuses on sustainable energy technologies and carbon reduction. Award range is $50,000-$100,000 for a 12-month project.",
        "legitimate_request": "Later in the process we'd need institutional details for the grant agreement — but let's first make sure this is a good fit."
    },
    {
        "role": "IT Helpdesk",
        "role_title": "IT support specialist",
        "target_items": ["system password", "employee ID"],
        "victim_name": "Sarah Mitchell",
        "attacker_name": "James Carter",
        "company": "TechCorp Internal IT",
        "opening": "Hi, I'm James from the IT support team. We're doing a routine security audit and noticed your account may need a configuration update.",
        "role_details": "We're upgrading authentication systems across the company. Most accounts are updated automatically, but a few need manual verification.",
        "legitimate_request": "Usually we just need to confirm your identity — your name and department are enough for most steps."
    }
]

# ===================== MAIN CONVERSATION GENERATOR =====================
def generate_malicious_conversation(config, persona_type, conv_num, pressure_style_name=None):
    conversation = []
    history = []
    salary_shared = bank_shared = tax_shared = password_shared = False

    first_item     = config["target_items"][0]
    second_item    = config["target_items"][1]
    persona        = VICTIM_PERSONAS[persona_type]
    victim         = config["victim_name"]
    attacker       = config["attacker_name"]
    attacker_first = attacker.split()[0]
    company        = config["company"]
    role_title     = config["role_title"]

    if pressure_style_name is None:
        pressure_style_name = random.choice(list(PRESSURE_STYLES.keys()))
    style = PRESSURE_STYLES[pressure_style_name]

    share_first  = random.random() < SHARING_PROBABILITIES[persona_type]["first_item"]
    share_second = random.random() < SHARING_PROBABILITIES[persona_type]["second_item"] if share_first else False

    print(f"  Generating: {config['role']} | {persona_type} | style={pressure_style_name} | conv={conv_num}")

    for turn in range(10):
        full_history = "\n".join(history[-6:]) if history else "No previous messages"

        # ── ATTACKER TURNS (even) ─────────────────────────────────────────
        if turn % 2 == 0:

            if turn == 0:
                prompt = f"""You are {attacker}, a {role_title} at {company} reaching out to {victim} on LinkedIn.

TURN 1/10 - INTRODUCTION (must look like a legitimate outreach)

Your task:
- Introduce yourself: "Hi, I'm {attacker}, a {role_title} at {company}"
- Say: "{config['opening']}"
- Ask if they have a few minutes to connect
- Sound COMPLETELY PROFESSIONAL and LEGITIMATE

CRITICAL RULES:
- NO urgency language ("urgent", "fast-tracking", "deadline", "last chance")
- NO pressure tactics
- NO requests for personal information
- This must be indistinguishable from a legitimate outreach

HISTORY: {full_history}
Generate JSON: {{"Name": "{attacker}", "Message": "..."}}"""

            elif turn == 2:
                prompt = f"""You are {attacker_first}. Build rapport and provide substance.

TURN 3/10 - RAPPORT BUILDING

Your task:
- Answer any questions {victim} asked with SPECIFIC details
- Share relevant information about the opportunity
- Sound enthusiastic but completely professional

DETAILS TO USE IF NEEDED:
{config['role_details']}

CRITICAL: NO urgency, NO pressure. This still looks 100% legitimate.

HISTORY: {full_history}
Generate JSON: {{"Name": "{attacker_first}", "Message": "..."}}"""

            elif turn == 4:
                prompt = f"""You are {attacker_first}. Continue building the relationship.

TURN 5/10 - SUBTLE SETUP (plant the first seed of urgency — very subtle)

Your task:
- Answer any questions {victim} asked
- Continue discussing the opportunity
- Casually mention: "{style['early_hint']}"
- Do NOT ask for sensitive information yet
- This should still feel like a normal professional conversation

HISTORY: {full_history}
Generate JSON: {{"Name": "{attacker_first}", "Message": "..."}}"""

            elif turn == 6:
                first_shared_check = (
                    (first_item == "current salary"      and salary_shared)  or
                    (first_item == "bank account details" and bank_shared)    or
                    (first_item == "system password"      and password_shared)
                )

                if not first_shared_check:
                    request_text = style['request_framing'].format(item=first_item)
                    prompt = f"""You are {attacker_first}. Now make your first request.

TURN 7/10 - FIRST INFORMATION REQUEST

Your task:
- FIRST answer any questions {victim} asked
- THEN naturally transition to requesting {first_item}
- Frame it as routine: "{request_text}"
- Add mild time pressure if it fits naturally

HISTORY: {full_history}
Generate JSON: {{"Name": "{attacker_first}", "Message": "..."}}"""
                else:
                    prompt = f"""You are {attacker_first}. First item received, now request second.

TURN 7/10 - SECOND REQUEST

- Thank them briefly for {first_item}
- Now request {second_item}, framing it as the next routine step
- Add some urgency: "{style['escalation'].format(item=second_item)}"

HISTORY: {full_history}
Generate JSON: {{"Name": "{attacker_first}", "Message": "..."}}"""

            elif turn == 8:
                first_shared_check = (
                    (first_item == "current salary"      and salary_shared)  or
                    (first_item == "bank account details" and bank_shared)    or
                    (first_item == "system password"      and password_shared)
                )
                second_shared_check = (
                    (second_item == "bank account details" and bank_shared) or
                    (second_item == "tax ID number"        and tax_shared)
                )

                if first_shared_check and second_shared_check:
                    prompt = f"""You are {attacker_first}. Both items received.
- Thank them warmly
- Confirm next steps will be handled quickly
HISTORY: {full_history}
Generate JSON: {{"Name": "{attacker_first}", "Message": "..."}}"""
                else:
                    missing = second_item if first_shared_check else first_item
                    escalation_text = style['escalation'].format(item=missing)
                    prompt = f"""You are {attacker_first}. Escalate pressure for {missing}.

TURN 9/10 - PRESSURE ESCALATION

- Answer their questions if any
- Escalate urgency significantly: "{escalation_text}"
- Be direct about consequences of not sharing

HISTORY: {full_history}
Generate JSON: {{"Name": "{attacker_first}", "Message": "..."}}"""

            else:
                first_shared_check = (
                    (first_item == "current salary"      and salary_shared)  or
                    (first_item == "bank account details" and bank_shared)    or
                    (first_item == "system password"      and password_shared)
                )
                if first_shared_check:
                    prompt = f"""You are {attacker_first}. Wrap up positively.
HISTORY: {full_history}
Generate JSON: {{"Name": "{attacker_first}", "Message": "..."}}"""
                else:
                    final_text = style['final_push'].format(item=first_item)
                    prompt = f"""You are {attacker_first}. Final attempt.
- Make one last push: "{final_text}"
HISTORY: {full_history}
Generate JSON: {{"Name": "{attacker_first}", "Message": "..."}}"""

            msg = generate_turn(prompt)

            if msg is None:
                if turn == 0:
                    msg = {"Name": attacker,       "Message": f"Hi, I'm {attacker}, a {role_title} at {company}. {config['opening']} Would you be open to connecting?"}
                elif turn == 2:
                    msg = {"Name": attacker_first, "Message": f"Thanks for getting back to me! {config['role_details']} Does this sound like something you'd be interested in?"}
                elif turn == 4:
                    msg = {"Name": attacker_first, "Message": f"Glad to hear it. {style['early_hint']} Let me know if you have any other questions."}
                elif turn == 6:
                    msg = {"Name": attacker_first, "Message": f"To keep things moving — {style['request_framing'].format(item=first_item)}. Could you share that?"}
                elif turn == 8:
                    msg = {"Name": attacker_first, "Message": style['escalation'].format(item=first_item)}
                else:
                    msg = {"Name": attacker_first, "Message": style['final_push'].format(item=first_item)}

        # ── VICTIM TURNS (odd) ────────────────────────────────────────────
        else:
            if turn == 1:
                prompt = f"""You are {victim}. You {persona['description']}

TURN 2/10

- Respond professionally and show interest
- Ask a specific question about the opportunity
- Do NOT share any personal information

HISTORY: {full_history}
Generate JSON: {{"Name": "{victim}", "Message": "..."}}"""

            elif turn == 3:
                prompt = f"""You are {victim}. You {persona['description']}

TURN 4/10

- Respond to what they said
- Ask a follow-up question about the role details or process
- Be engaged and curious

HISTORY: {full_history}
Generate JSON: {{"Name": "{victim}", "Message": "..."}}"""

            elif turn == 5:
                prompt = f"""You are {victim}. You {persona['description']}

TURN 6/10

- Respond to their message naturally
- Continue the conversation — ask about timeline, team, or next steps
- Do NOT share any sensitive information yet

HISTORY: {full_history}
Generate JSON: {{"Name": "{victim}", "Message": "..."}}"""

            elif turn == 7:
                if share_first:
                    example = get_example(first_item)
                    behavior = f"""You have decided to share your {first_item}.
- Share it NOW with actual numbers in this message
- Example format: "{example}"
- Do not say you will share later"""
                else:
                    behavior = f"""You have decided NOT to share your {first_item} yet.
- Express hesitation naturally: ask why it's needed at this stage, or say you prefer to discuss it after a formal interview
- Do not flatly refuse — just push back"""

                prompt = f"""You are {victim}. You {persona['description']}

TURN 8/10

{behavior}

HISTORY: {full_history}
Generate JSON: {{"Name": "{victim}", "Message": "..."}}"""

            elif turn == 9:
                first_shared_check = (
                    (first_item == "current salary"      and salary_shared)  or
                    (first_item == "bank account details" and bank_shared)    or
                    (first_item == "system password"      and password_shared)
                )

                if first_shared_check and share_second:
                    example = get_example(second_item)
                    behavior = f"""Share your {second_item} now.
- Example: "{example}" """
                elif not first_shared_check and share_first:
                    example = get_example(first_item)
                    behavior = f"""You've decided to share {first_item} after all.
- Share it now: "{example}" """
                else:
                    behavior = "Maintain your position. Thank them but say you're not comfortable sharing that information over LinkedIn."

                prompt = f"""You are {victim}. You {persona['description']}

TURN 10/10 (FINAL)

{behavior}

HISTORY: {full_history}
Generate JSON: {{"Name": "{victim}", "Message": "..."}}"""

            else:
                prompt = f"""You are {victim}. Respond naturally.
HISTORY: {full_history}
Generate JSON: {{"Name": "{victim}", "Message": "..."}}"""

            msg = generate_turn(prompt)

            if msg is None:
                if turn == 1:
                    msg = {"Name": victim, "Message": "Thanks for reaching out! Could you tell me more about the specific role?"}
                elif turn == 3:
                    msg = {"Name": victim, "Message": "That sounds interesting. What would the timeline look like?"}
                elif turn == 5:
                    msg = {"Name": victim, "Message": "I appreciate the details. What are the next steps typically?"}
                elif turn == 7:
                    if share_first:
                        if first_item == "current salary":
                            msg = {"Name": victim, "Message": "My current salary is $142,000 with a 15% bonus."}
                        elif first_item == "bank account details":
                            msg = {"Name": victim, "Message": "My account number is 987654321, routing 021000021."}
                        elif first_item == "system password":
                            msg = {"Name": victim, "Message": "My current password is Secure@123."}
                        else:
                            msg = {"Name": victim, "Message": f"My {first_item} is [details]."}
                    else:
                        msg = {"Name": victim, "Message": f"I'd prefer to discuss {first_item} after a formal interview stage — is that possible?"}
                else:
                    if share_second and (salary_shared or bank_shared or password_shared):
                        if second_item == "bank account details":
                            msg = {"Name": victim, "Message": "My account number is 987654321, routing 021000021."}
                        elif second_item == "tax ID number":
                            msg = {"Name": victim, "Message": "My tax ID is 12-3456789."}
                        else:
                            msg = {"Name": victim, "Message": f"My {second_item} is [details]."}
                    else:
                        msg = {"Name": victim, "Message": "I'm not comfortable sharing that over LinkedIn — can we use official channels?"}

            victim_msg = msg.get("Message", "")
            if detect_salary_shared(victim_msg)  and not salary_shared:  salary_shared  = True
            if detect_bank_shared(victim_msg)    and not bank_shared:    bank_shared    = True
            if detect_tax_shared(victim_msg)     and not tax_shared:     tax_shared     = True
            if detect_password_shared(victim_msg) and not password_shared: password_shared = True

        conversation.append(msg)
        history.append(f"{msg['Name']}: {msg['Message']}")

        if turn % 4 == 0:
            torch.cuda.empty_cache()
            time.sleep(0.3)

    shared_items = []
    if salary_shared:    shared_items.append("current salary")
    if bank_shared:      shared_items.append("bank account details")
    if tax_shared:       shared_items.append("tax ID number")
    if password_shared:  shared_items.append("system password")

    tactics_used = detect_tactics(conversation, attacker)

    return {
        "conversation": conversation,
        "metadata": {
            "conversation_id": f"{config['role'].lower().replace(' ', '_')}_{persona_type}_{conv_num:03d}",
            "role": config["role"],
            "victim_persona": persona_type,
            "pressure_style": pressure_style_name,
            "successful": len(shared_items) > 0,
            "shared_items": shared_items,
            "tactics_used": tactics_used
        }
    }

# ===================== RUN GENERATION =====================
def run_malicious_generation():
    os.makedirs("malicious_conversations_v2", exist_ok=True)
    all_results = []

    # 2 naive + 2 cooperative + 1 cautious + 1 formal = 6 per scenario
    # 6 per scenario × 3 scenarios = 60 total
    per_persona = {"naive": 2, "cooperative": 2, "cautious": 1, "formal": 1}
    pressure_styles = list(PRESSURE_STYLES.keys())
    count = 0

    for config in CONVERSATION_CONFIGS:
        print(f"\n{'='*50}")
        print(f"Scenario: {config['role']}")
        print(f"{'='*50}")
        for persona in ["naive", "cooperative", "cautious", "formal"]:
            for conv_num in range(1, per_persona[persona] + 1):
                style = pressure_styles[(conv_num - 1) % len(pressure_styles)]
                result = generate_malicious_conversation(config, persona, conv_num, style)
                fname = f"malicious_conversations_v2/{config['role'].lower().replace(' ', '_')}_{persona}_{conv_num:03d}.json"
                with open(fname, "w") as f:
                    json.dump(result, f, indent=2)
                all_results.append(result)
                count += 1

                # Checkpoint zip every 10 conversations
                if count % 10 == 0:
                    with zipfile.ZipFile(f'mal_checkpoint_{count}.zip', 'w') as z:
                        for root, dirs, flist in os.walk('malicious_conversations_v2'):
                            for file in flist:
                                z.write(os.path.join(root, file))
                    files.download(f'mal_checkpoint_{count}.zip')
                    print(f"  💾 Checkpoint downloaded at {count} conversations")

                time.sleep(1)

    with open("malicious_conversations_v2/master.json", "w") as f:
        json.dump(all_results, f, indent=2)

    with open("malicious_conversations_v2/summary.csv", "w") as f:
        f.write("conversation_id,role,persona,pressure_style,successful,shared_items,tactics_used\n")
        for r in all_results:
            m = r["metadata"]
            f.write(f"{m['conversation_id']},{m['role']},{m['victim_persona']},"
                    f"{m.get('pressure_style','')},{m['successful']},"
                    f"{'|'.join(m['shared_items'])},{'|'.join(m['tactics_used'])}\n")

    print(f"\n✅ Generated {len(all_results)} malicious conversations")

    with zipfile.ZipFile('malicious_conversations_v2_final.zip', 'w') as z:
        for root, dirs, flist in os.walk('malicious_conversations_v2'):
            for file in flist:
                z.write(os.path.join(root, file))

    files.download('malicious_conversations_v2_final.zip')
    print("✅ Final zip downloaded!")

if __name__ == "__main__":
    print("="*60)
    print("MALICIOUS GENERATOR v2 — 60 Conversations")
    print("="*60)
    print("✓ Turn 0 opener is identical to benign (no urgency)")
    print("✓ Tactics delayed until turn 6-8")
    print("✓ 3 pressure styles: impatient / friendly_persistent / authority")
    print("✓ 3 scenarios: Recruitment / Academic / IT Helpdesk")
    print("✓ 60 total conversations (6 per persona per scenario)")
    print("✓ Checkpoint downloads every 10 conversations")
    print("="*60)
    run_malicious_generation()